In [37]:
import requests
from bs4 import BeautifulSoup
import re
import pandas as pd
from io import StringIO
from PIL import Image
from io import BytesIO

In [2]:
session = requests.Session()

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.5',
    'Referer': 'https://www.google.com',
    'Connection': 'keep-alive'
}

session.headers.update(headers)

In [267]:
def scrap_url(url, session):

    try:
    
        response = session.get(url)
        soup = BeautifulSoup(response.text, parser='lxml',features='lxml')
    
        product_name_tag = soup.find('div', class_='tb_wt tb_wt_page_title_system tb_mb_10 display-block tb_system_page_title')
        product_name = product_name_tag.text if product_name_tag else 'Unknown'

        
    
        info_tag = soup.find('div', id='ProductInfoSystem_Ho7r8pnm')
        in_stock, product_code, brand = False, 'Unknown', 'Unknown'
        
        if info_tag:
            info_list = [info.text for info in info_tag.find_all('dd')]
            in_stock = True if info_list[0] == 'In Stock' else False
            product_code = info_list[1]
            brand = info_list[2]
            


        original_price = current_price = None
        currency = 'Unknown'
        price_tag = soup.find('div', class_='price')

        if price_tag:
            price_regular_tag = price_tag.find('span', class_='price-regular')
            original_price_tag = price_tag.find('span', class_='price-old')
            
            if price_regular_tag:
                current_price = float(price_regular_tag.text.split()[1].replace(',',''))
                
            elif original_price_tag:
                    original_price = float(original_price_tag.text.split()[1].replace(',',''))
                
                    current_price_tag = price_tag.find('span', class_='price-new')
                    current_price = float(current_price_tag.text.split()[1].replace(',',''))
                    
            currency =  price_tag.text.split()[0]
            

        
        product_description_tag = soup.find('div', id='ProductDescriptionSystem_Bx7ALQdc')
        product_description = 'Unknown'
        
        if product_description_tag:
            p_tags = product_description_tag.find_all('p')
            product_description_list = [p.text for p in p_tags if p.text.strip()]
            product_description = '\n'.join(product_description_list)


        
        table_tag = soup.find_all('div', id='Text_N6iK9e7t')
        for tt in table_tag:
            tt.decompose()
    
        found_tables = soup.find('table')
        properties = {}
        
        if found_tables:
            df = pd.read_html(StringIO(soup.prettify()))[0]
            df.columns = ['Specification', 'Value']
            properties = {l[0] : l[1] for l in df.to_numpy()}
    
    
        thumbnail_tag = soup.find('a', class_='thumbnail')
        image_url = thumbnail_tag.get('href') if thumbnail_tag else None
    
        product_dict = {
            'product_code' : product_code,
            'product_name' : product_name,
            'brand' : brand,
            'store' : 'Compu Jordan',
            'url' : url,
            'image_url' : image_url,
    
            'price' : current_price,
            'original_price' : original_price,
            'currency' : currency,
            'in_stock' : in_stock,
    
            'product_description' : product_description,
    
            'categories' : {},
            'specs' : properties
        }
    
        return product_dict

    except Exception as e:
        print(f'failed to scrap {url} : {e}')
        return None

In [223]:
url = input()

 https://compujordan.com/asus-tuf-gaming-geforce-rtx-4060-ti-8gb-gddr6-oc-edition-gaming-graphics-card


In [224]:
response = session.get(url)

In [225]:
soup = BeautifulSoup(response.text, parser='lxml',features='lxml')

In [226]:
product_name_tag = soup.find('div', class_='tb_wt tb_wt_page_title_system tb_mb_10 display-block tb_system_page_title')
product_name = product_name_tag.text

In [227]:
product_name

'ASUS TUF Gaming GeForce RTX 4060 Ti 8GB GDDR6 OC Edition Gaming Graphics Card'

In [228]:
info_tag = soup.find('div', id='ProductInfoSystem_Ho7r8pnm')

In [229]:
info_list = [info.text for info in info_tag.find_all('dd')]

In [257]:
in_stock = True if info_list[0] == 'In Stock' else False

In [231]:
product_code = info_list[1]

In [232]:
brand = info_list[2]

In [233]:
price_tag = soup.find('div', class_='price')

price_regular_tag = price_tag.find('span', class_='price-regular')
original_price_tag = price_tag.find('span', class_='price-old')


original_price = current_price = None

if price_regular_tag:
    current_price = float(price_regular_tag.text.split()[1].replace(',',''))
    
elif original_price_tag:
        original_price = float(original_price_tag.text.split()[1].replace(',',''))
    
        current_price_tag = price_tag.find('span', class_='price-new')
        current_price = float(current_price_tag.text.split()[1].replace(',',''))
        
    
currency =  price_tag.text.split()[0]

In [234]:
print(f'{current_price = }, {original_price = }, {currency = }')

current_price = 379.0, original_price = 419.0, currency = 'JOD'


In [235]:
product_description_tag = soup.find('div', id='ProductDescriptionSystem_Bx7ALQdc')
product_description = product_description_tag.text

In [236]:
p_tags = product_description_tag.find_all('p')

In [237]:
product_descriptions = [p.text for p in p_tags if p.text.strip()]

In [238]:
product_descriptions

['The GeForce RTX series is produced by NVIDIA, a leading manufacturer of graphics processing units (GPUs) for gaming and professional applications. The "TUF Gaming" branding from ASUS typically indicates that the graphics card is designed to withstand demanding gaming conditions, with features such as robust cooling solutions and durable components.',
 'Here are some general expectations you might have for a graphics card like the ASUS TUF Gaming GeForce RTX 4060 Ti:',
 'RTX Series Features: The RTX series graphics cards from NVIDIA typically support real-time ray tracing and AI-enhanced graphics rendering through technologies such as NVIDIA DLSS (Deep Learning Super Sampling).',
 'Performance: The "Ti" designation often indicates a higher-performing variant within a particular GPU series. Therefore, you can expect solid gaming performance at high resolutions and detail settings.',
 '8GB GDDR6 Memory: The 8GB of GDDR6 memory suggests that the graphics card should be capable of handlin

In [239]:
table_tag = soup.find_all('div', id='Text_N6iK9e7t')

In [240]:
for tt in table_tag:
    tt.decompose()

In [241]:
found_tables = soup.find('table')

In [242]:
print(type(found_tables) == None)

False


In [243]:
if found_tables:
    df = pd.read_html(StringIO(soup.prettify()))[0]
    df.columns = ['Specification', 'Value']
    properties = {l[0] : l[1] for l in df.to_numpy()}

properties

{'Graphic Engine': 'NVIDIA® GeForce RTX™ 4060 Ti',
 'Bus Standard': 'PCI Express 4.0',
 'OpenGL': 'OpenGL®4.6',
 'Video Memory': '8GB GDDR6',
 'Engine Clock': 'OC Mode: 2655MHz Default Mode: 2625MHz (Boost)',
 'CUDA Core': '4352',
 'Memory Speed': '18 Gbps',
 'Memory Interface': '128-bit',
 'Resolution': 'Digital Max Resolution 7680 x 4320',
 'Interface': 'Yes x 1 (Native HDMI 2.1a)  Yes x 3 (Native DisplayPort 1.4a)  HDCP Support Yes (2.3)',
 'Maximum Display Support': '4',
 'NVlink/ Crossfire Support': 'NO',
 'Accessories': '1 x Collection Card 1 x Speedsetup Manual 1 x Thank you Card 1 x TUF Graphic card holder 1 x TUF Gaming Ceritificate 1 x TUF Velcro Hook & Loop',
 'Software': 'ASUS GPU Tweak III & GeForce Game Ready Driver & Studio Driver: please download all software from the support site.',
 'Recommended PSU': '650W',
 'Power Connectors': '1 x 8-pin',
 'Slot': '3.12 slot',
 'AURA SYNC': 'ARGB',
 'Warranty': '1 Year',
 'Width X Depth X Height': '300.00mm x 139.00mm x 62.40mm'}

In [244]:
with open('../../../data/compujordan/test.html', 'w') as f:
    f.write(response.text)

In [245]:
soup.find('img')

<img height="1" src="https://www.facebook.com/tr?id=158080995626152&amp;ev=PageView&amp;noscript=1" style="display:none" width="1"/>

In [246]:
img_tag = soup.find('img', class_='zoomImg')

In [247]:
type(img_tag)

NoneType

In [248]:
thumbnail_tag = soup.find('a', class_='thumbnail')

In [249]:
image_url = thumbnail_tag.get('href')

In [250]:
image_response = session.get(image_url)

In [251]:
img = Image.open(BytesIO(image_response.content))

In [252]:
breadcrumb_ul = soup.find('ul', class_='breadcrumb')

In [253]:
breadcrumbs_li = [li.text.strip() for li in breadcrumb_ul.find_all('li')]

In [254]:
categories = breadcrumbs_li[1:-1]

In [255]:
categories

[]

In [256]:
breadcrumbs_li

['Home',
 'ASUS TUF Gaming GeForce RTX 4060 Ti 8GB GDDR6 OC Edition Gaming Graphics Card']

In [273]:
d = scrap_url('https://citycenter.jo/gaming/gaming-gaming-laptops/lenovo-new-loq-gaming-2025-13gen-intel-core-i7-13650hx-14-cores-w-ai-engine-la1-chip-nvidia-rtx-5060-8gb-ddr7-ips-144hz-display-windows-11-luna-grey', session)

In [274]:
d

{'product_code': ' 83JE002LUS',
 'product_name': 'Lenovo NEW LOQ Gaming (2025) 13Gen Intel Core i7 13650HX 14-Cores w/ AI Engine+ LA1 Chip & Nvidia RTX 5060 8GB DDR7 & IPS 144Hz Display & Windows 11 - Luna Grey',
 'brand': 'Lenovo',
 'store': 'Compu Jordan',
 'url': 'https://citycenter.jo/gaming/gaming-gaming-laptops/lenovo-new-loq-gaming-2025-13gen-intel-core-i7-13650hx-14-cores-w-ai-engine-la1-chip-nvidia-rtx-5060-8gb-ddr7-ips-144hz-display-windows-11-luna-grey',
 'image_url': 'https://image.citycenter.jo/cachewebp/catalog/002025/72025/LQ1572025-1200x1200.webp',
 'price': 899.0,
 'original_price': 949.0,
 'currency': 'JOD',
 'in_stock': True,
 'product_description': 'Lenovo\nLOQ 15IRX10',
 'categories': {},
 'specs': {'Name': nan,
  'Company Name': nan,
  'Email': nan,
  'Phone': nan,
  nan: nan}}